# 🦜 BirdCLEF+ 2026 — Submission Notebook

**Competition:** Brazil Pantanal wildlife species identification from audio

**Aapka kaam:** Har 5-second audio clip ke liye 234 species ki probability predict karna

**Strategy:** 
- Audio ko Mel Spectrogram image mein convert karo
- Pretrained EfficientNet model use karo (fast + accurate)
- Har 5-second window ke liye species probabilities predict karo

---
**Cells ka order:**
1. Libraries install/import
2. Config (settings)
3. Data paths check
4. Audio → Spectrogram function
5. Dataset class
6. Model banana
7. Training
8. Inference (test data predict karna)
9. Submission file banana


## Cell 1: Libraries Install karo

Yahan woh tools install hote hain jo audio process karne ke liye chahiye.
- `librosa` = audio file padhne aur spectrogram banane ke liye
- `timm` = pretrained image models (EfficientNet) ke liye
- `audioread` = audio backend

In [1]:
# Zaruri libraries install karo
# Kaggle pe yeh pehle se available hoti hain, lekin ensure karne ke liye:
import subprocess
subprocess.run(['pip', 'install', 'librosa', 'timm', '--quiet'], capture_output=True)

CompletedProcess(args=['pip', 'install', 'librosa', 'timm', '--quiet'], returncode=0, stdout=b'', stderr=b'')

## Cell 2: Sari Libraries Import karo

- `torch` = PyTorch (deep learning)
- `librosa` = audio processing
- `timm` = pretrained models
- `numpy/pandas` = data handle karna

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Sari libraries import ho gayi!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✅ Sari libraries import ho gayi!
PyTorch version: 2.10.0+cpu
CUDA available: False


## Cell 3: Configuration (Settings)

Yahan saari important settings ek jagah set hoti hain:
- `SAMPLE_RATE` = audio kitne samples per second (32kHz competition ka standard)
- `DURATION` = kitne seconds ki audio ek baar process karni hai (5 sec = competition requirement)
- `N_MELS` = spectrogram ki height (frequency bands)
- `NUM_CLASSES` = 234 species (competition mein itni species hain)
- `EPOCHS` = model kitni baar training data dekhe

In [3]:
class CFG:
    # ── Audio Settings ──────────────────────────────
    SAMPLE_RATE = 32000      # 32kHz - competition ka standard
    DURATION    = 5          # 5 seconds per clip (competition requirement)
    N_SAMPLES   = SAMPLE_RATE * DURATION  # = 160,000 samples per clip
    
    # ── Spectrogram Settings ─────────────────────────
    N_FFT       = 1024       # FFT window size
    HOP_LENGTH  = 320        # FFT hop (overlap control)
    N_MELS      = 128        # Mel frequency bands (spectrogram height)
    FMIN        = 20         # Minimum frequency (Hz) - human hearing start
    FMAX        = 16000      # Maximum frequency (Hz) - 32kHz/2
    
    # ── Model Settings ───────────────────────────────
    MODEL_NAME  = 'efficientnet_b0'  # Pretrained model (fast, CPU friendly)
    NUM_CLASSES = 234               # 234 species in this competition
    
    # ── Training Settings ────────────────────────────
    BATCH_SIZE  = 16         # Ek baar kitne samples process hon
    EPOCHS      = 5          # Kitni baar poora data dekhe (CPU time limit hai)
    LR          = 1e-3       # Learning rate (model kitni tezi se seekhe)
    
    # ── Paths (Kaggle ka structure) ──────────────────
    BASE_PATH        = Path('/kaggle/input/competitions/birdclef-2026')
    TRAIN_AUDIO_DIR  = BASE_PATH / 'train_audio'
    TRAIN_SOUNDS_DIR = BASE_PATH / 'train_soundscapes'
    TEST_DIR         = BASE_PATH / 'test_soundscapes'
    TRAIN_CSV        = BASE_PATH / 'train.csv'
    TAXONOMY_CSV     = BASE_PATH / 'taxonomy.csv'
    SAMPLE_SUB       = BASE_PATH / 'sample_submission.csv'
    OUTPUT_DIR       = Path('/kaggle/working')
    
    # ── Device ──────────────────────────────────────
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"✅ Config ready!")
print(f"Device: {CFG.DEVICE}")
print(f"Audio: {CFG.DURATION}s @ {CFG.SAMPLE_RATE}Hz = {CFG.N_SAMPLES:,} samples")
print(f"Species count: {CFG.NUM_CLASSES}")

✅ Config ready!
Device: cpu
Audio: 5s @ 32000Hz = 160,000 samples
Species count: 234


## Cell 4: Data Files Check karo

Dekho ke saari files sahi jagah hain ya nahin.
- `train.csv` mein species labels hain
- `taxonomy.csv` mein 234 species ki list hai
- `sample_submission.csv` mein submission format hai

In [4]:
import pandas as pd
from pathlib import Path

# SAHI PATH — competitions folder mein hai
BASE = Path('/kaggle/input/competitions/birdclef-2026')

train_csv    = BASE / 'train.csv'
taxonomy_csv = BASE / 'taxonomy.csv'
sample_csv   = BASE / 'sample_submission.csv'
train_audio  = BASE / 'train_audio'
test_dir     = BASE / 'test_soundscapes'

files = [train_csv, taxonomy_csv, sample_csv, train_audio, test_dir]
print("File check:")
for f in files:
    print("  OK" if f.exists() else "  MISSING", f.name)

train_df   = pd.read_csv(train_csv)
taxonomy   = pd.read_csv(taxonomy_csv)
sample_sub = pd.read_csv(sample_csv)

print(f"Training rows:  {len(train_df):,}")
print(f"Species count:  {len(taxonomy)}")
print(f"Submission rows: {len(sample_sub):,}")
train_df.head(3)

File check:
  OK train.csv
  OK taxonomy.csv
  OK sample_submission.csv
  OK train_audio
  OK test_soundscapes
Training rows:  35,549
Species count:  234
Submission rows: 3


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat


## Cell 5: Species List aur Label Mapping

Model ko numbers samajh mein aate hain, species names nahin.
Is liye har species code ko ek number (index) dete hain:
- `label2idx`: species code → number  (training ke liye)
- `idx2label`: number → species code  (prediction ke baad)

**Example:** `brnowl` → `42`, `42` → `brnowl`

In [5]:
# Sample submission se species columns nikalo
# Pehla column 'row_id' hai, baaki sab species hain
SPECIES_COLS = [c for c in sample_sub.columns if c != 'row_id']
NUM_CLASSES  = len(SPECIES_COLS)

# Label ↔ Index mapping banao
label2idx = {label: i for i, label in enumerate(SPECIES_COLS)}
idx2label = {i: label for label, i in label2idx.items()}

print(f"✅ Species: {NUM_CLASSES}")
print(f"\nPehli 5 species:")
for i in range(5):
    print(f"  [{i}] {SPECIES_COLS[i]}")

# Train data mein sirf woh recordings rakho jinki species submission mein hai
train_df = train_df[train_df['primary_label'].isin(label2idx)].reset_index(drop=True)
train_df['label_idx'] = train_df['primary_label'].map(label2idx)

print(f"\n✅ Filtered training data: {len(train_df):,} recordings")

✅ Species: 234

Pehli 5 species:
  [0] 1161364
  [1] 116570
  [2] 1176823
  [3] 1491113
  [4] 1595929

✅ Filtered training data: 35,549 recordings


## Cell 6: Audio → Mel Spectrogram Function

**Spectrogram kya hota hai?**
Audio ek wave hai — upar neeche oscillations. Spectrogram us wave ko image mein convert karta hai:
- X-axis = time (waqt)
- Y-axis = frequency (awaaz ki pitch)
- Color = volume (kitni tez awaaz)

Har chiriya/janwar ka spectrogram unique hota hai — jaise fingerprint.
Model phir is image ko dekh ke pehchanta hai ke kaunsi species hai.

**Mel scale** = human ear ki tarah frequency dekhna (low frequencies mein zyada detail)

In [6]:
def load_audio(filepath, sr=CFG.SAMPLE_RATE, duration=CFG.DURATION):
    """
    Audio file load karo aur fix length mein cut/pad karo.
    
    Args:
        filepath: Audio file ka path
        sr: Sample rate (32000)
        duration: Kitne seconds chahiye (5)
    
    Returns:
        numpy array of audio samples (length = sr * duration)
    """
    target_len = sr * duration
    
    try:
        # Audio load karo (librosa automatically mono karta hai)
        audio, _ = librosa.load(filepath, sr=sr, mono=True, duration=duration + 1)
        
        if len(audio) >= target_len:
            # Zyada lamba hai: beech se target_len utna kaat lo
            start = (len(audio) - target_len) // 2
            audio = audio[start : start + target_len]
        else:
            # Chota hai: zeros se pad karo (silence add karo)
            pad_len = target_len - len(audio)
            audio = np.pad(audio, (0, pad_len), mode='constant')
        
        return audio.astype(np.float32)
    
    except Exception as e:
        # Koi problem ho to zeros return karo (silent audio)
        print(f"Warning: {filepath} load nahin hua: {e}")
        return np.zeros(target_len, dtype=np.float32)


def audio_to_melspectrogram(audio, sr=CFG.SAMPLE_RATE):
    """
    Audio array ko Mel Spectrogram mein convert karo.
    
    Steps:
    1. Mel spectrogram compute karo
    2. dB scale mein convert karo (log scale, human ear jaisi)
    3. Normalize karo (0 to 1 range)
    4. 3 channels banao (RGB jaisa, model ke liye)
    
    Returns:
        Tensor shape: (3, N_MELS, time_frames)
    """
    # Step 1: Mel spectrogram
    mel = librosa.feature.melspectrogram(
        y          = audio,
        sr         = sr,
        n_fft      = CFG.N_FFT,
        hop_length = CFG.HOP_LENGTH,
        n_mels     = CFG.N_MELS,
        fmin       = CFG.FMIN,
        fmax       = CFG.FMAX,
    )
    
    # Step 2: dB mein convert (log scale)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    # Step 3: Normalize to [0, 1]
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    
    # Step 4: 3 channels banao (EfficientNet RGB expect karta hai)
    mel_3ch = np.stack([mel_norm, mel_norm, mel_norm], axis=0)  # shape: (3, 128, T)
    
    return mel_3ch.astype(np.float32)


print("✅ Audio processing functions ready!")
print(f"   Spectrogram shape hogi: (3, {CFG.N_MELS}, ~{CFG.N_SAMPLES // CFG.HOP_LENGTH})")

✅ Audio processing functions ready!
   Spectrogram shape hogi: (3, 128, ~500)


## Cell 7: Training Dataset Class

PyTorch ko data ek specific format mein chahiye.
`BirdDataset` class batati hai:
- Kahan se audio file load karni hai
- Usse spectrogram mein kaise convert karna hai
- Kaunsa label (species index) return karna hai

Yeh class ek library jaisi hai — model jab bhi koi sample maange, yeh usse process karke deti hai.

In [7]:
class BirdTrainDataset(Dataset):
    """
    Training Dataset:
    - Audio file load karo
    - Spectrogram banao
    - Label return karo
    """
    def __init__(self, df, audio_dir):
        self.df        = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
    
    def __len__(self):
        """Kitne samples hain total"""
        return len(self.df)
    
    def __getitem__(self, idx):
        """idx-wa sample return karo"""
        row      = self.df.iloc[idx]
        filepath = self.audio_dir / row['filename']
        label    = int(row['label_idx'])
        
        # Audio load karo
        audio = load_audio(filepath)
        
        # Spectrogram banao
        spec = audio_to_melspectrogram(audio)
        
        # Tensor mein convert karo
        spec_tensor = torch.tensor(spec, dtype=torch.float32)
        
        return spec_tensor, torch.tensor(label, dtype=torch.long)


# Dataset banao — pehle 3000 samples se shuru karo (speed ke liye)
# Zyada accuracy chahiye to yeh number badhao
TRAIN_LIMIT = min(3000, len(train_df))  # CPU time limit 90 min hai
train_subset = train_df.sample(TRAIN_LIMIT, random_state=42).reset_index(drop=True)

train_dataset = BirdTrainDataset(train_subset, CFG.TRAIN_AUDIO_DIR)
train_loader  = DataLoader(
    train_dataset,
    batch_size  = CFG.BATCH_SIZE,
    shuffle     = True,       # Har epoch mein mix karo
    num_workers = 2,          # 2 CPU cores se data load karo parallel
    pin_memory  = False,
)

print(f"✅ Training dataset ready!")
print(f"   Samples: {len(train_dataset):,}")
print(f"   Batches per epoch: {len(train_loader)}")
print(f"   Batch size: {CFG.BATCH_SIZE}")

# Ek sample check karo
sample_spec, sample_label = train_dataset[0]
print(f"\n   Sample spectrogram shape: {sample_spec.shape}")
print(f"   Sample label: {sample_label} → {SPECIES_COLS[sample_label.item()]}")

✅ Training dataset ready!
   Samples: 3,000
   Batches per epoch: 188
   Batch size: 16

   Sample spectrogram shape: torch.Size([3, 128, 501])
   Sample label: 160 → purjay1


## Cell 8: Model Banana

**EfficientNet_b0 kya hai?**
Yeh ek pretrained model hai jo pehle se lakhon images dekh chuka hai (ImageNet).
Hum sirf iska last layer (head) replace karte hain apne 234 species ke liye.

**Transfer Learning:** Naye kaam ke liye pehle se trained brain use karna.
Jaise ek experienced insaan nayi cheez jaldi seekhta hai.

**Kyun EfficientNet_b0?**
- CPU pe bhi fast chalta hai
- Accuracy aur speed ka acha balance
- 90-minute time limit mein fit

In [8]:
class BirdCLEFModel(nn.Module):
    """
    BirdCLEF Model:
    - EfficientNet_b0 backbone (pretrained)
    - Custom head: 234 species ke liye
    """
    def __init__(self, num_classes=CFG.NUM_CLASSES, model_name=CFG.MODEL_NAME):
        super().__init__()
        
        # Pretrained EfficientNet load karo
        # num_classes=0 matlab last classification layer hata do
        self.backbone = timm.create_model(
            model_name,
            pretrained   = False,     # ← YAHAN SIRF TRUE se FALSE kiya hai
            num_classes  = 0,         # Features nikalne ke liye
            in_chans     = 3,         # 3 channels (RGB jaisa spectrogram)
        )
        
        # Backbone ki output features kitni hain?
        n_features = self.backbone.num_features  # EfficientNet_b0 = 1280
        
        # Custom classification head
        self.head = nn.Sequential(
            nn.Dropout(0.3),           # Overfitting rokne ke liye
            nn.Linear(n_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),  # 234 species output
        )
    
    def forward(self, x):
        """
        Forward pass:
        1. Spectrogram → backbone → features
        2. Features → head → logits (raw scores)
        """
        features = self.backbone(x)   # Shape: (batch, 1280)
        logits   = self.head(features) # Shape: (batch, 234)
        return logits


# Model banao aur device pe bhejo
model = BirdCLEFModel(num_classes=NUM_CLASSES).to(CFG.DEVICE)

# Parameters count karo
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model ready!")
print(f"   Model: {CFG.MODEL_NAME}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Device: {CFG.DEVICE}")

# Quick shape test
dummy = torch.randn(2, 3, CFG.N_MELS, 500).to(CFG.DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"   Test output shape: {out.shape}  (batch=2, classes={NUM_CLASSES})")
      

✅ Model ready!
   Model: efficientnet_b0
   Total parameters: 4,783,462
   Trainable parameters: 4,783,462
   Device: cpu
   Test output shape: torch.Size([2, 234])  (batch=2, classes=234)


## Cell 9: Training Loop

**Training kaise kaam karti hai:**
1. Model ek batch audio (spectrogram) dekhe
2. Predict kare ke kaunsi species hai
3. Galti (loss) calculate ho
4. Model galti se seekhe (backpropagation)
5. Repeat karo sare data ke liye (ek epoch)
6. Kai epochs tak repeat karo

**Loss function:** CrossEntropy — batata hai prediction kitna galat tha
**Optimizer:** Adam — model ke weights kaise update hon
**Scheduler:** Learning rate gradually kam karo (fine-tuning)

In [9]:
# Training components
criterion = nn.CrossEntropyLoss()          # Loss function
optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = CFG.LR,
    weight_decay = 1e-4,                   # L2 regularization
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max = CFG.EPOCHS,
    eta_min = 1e-6,
)


def train_one_epoch(model, loader, optimizer, criterion, epoch):
    """Ek epoch ki training"""
    model.train()  # Training mode on
    total_loss  = 0.0
    correct     = 0
    total       = 0
    
    for batch_idx, (specs, labels) in enumerate(loader):
        # Data device pe bhejo
        specs  = specs.to(CFG.DEVICE)
        labels = labels.to(CFG.DEVICE)
        
        # Forward pass
        optimizer.zero_grad()      # Purani gradients clear karo
        outputs = model(specs)     # Prediction karo
        loss    = criterion(outputs, labels)  # Loss calculate karo
        
        # Backward pass (seekhna)
        loss.backward()            # Gradients calculate karo
        optimizer.step()           # Weights update karo
        
        # Stats track karo
        total_loss += loss.item()
        preds       = outputs.argmax(dim=1)  # Highest probability wali class
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        
        # Har 10 batches pe progress print karo
        if (batch_idx + 1) % 10 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            acc      = 100.0 * correct / total
            print(f"  Epoch {epoch} | Batch {batch_idx+1}/{len(loader)} "
                  f"| Loss: {avg_loss:.4f} | Acc: {acc:.1f}%")
    
    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


# ── Training Start ──────────────────────────────────────────────
print("🚀 Training shuru ho rahi hai...")
print(f"   Epochs: {CFG.EPOCHS}")
print(f"   Batches per epoch: {len(train_loader)}")
print("-" * 60)

best_loss = float('inf')

for epoch in range(1, CFG.EPOCHS + 1):
    print(f"\n📍 Epoch {epoch}/{CFG.EPOCHS}")
    
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, epoch
    )
    scheduler.step()
    
    print(f"  ✅ Epoch {epoch} complete → Loss: {train_loss:.4f} | Acc: {train_acc:.1f}%")
    
    # Best model save karo
    if train_loss < best_loss:
        best_loss = train_loss
        torch.save(model.state_dict(), CFG.OUTPUT_DIR / 'best_model.pth')
        print(f"  💾 Best model saved (loss: {best_loss:.4f})")

print("\n✅ Training complete!")

🚀 Training shuru ho rahi hai...
   Epochs: 5
   Batches per epoch: 188
------------------------------------------------------------

📍 Epoch 1/5
  Epoch 1 | Batch 10/188 | Loss: 5.4385 | Acc: 1.2%
  Epoch 1 | Batch 20/188 | Loss: 5.3733 | Acc: 0.6%
  Epoch 1 | Batch 30/188 | Loss: 5.2857 | Acc: 0.6%
  Epoch 1 | Batch 40/188 | Loss: 5.2338 | Acc: 1.4%
  Epoch 1 | Batch 50/188 | Loss: 5.1995 | Acc: 1.8%
  Epoch 1 | Batch 60/188 | Loss: 5.1810 | Acc: 1.6%
  Epoch 1 | Batch 70/188 | Loss: 5.1726 | Acc: 2.0%
  Epoch 1 | Batch 80/188 | Loss: 5.1540 | Acc: 1.7%
  Epoch 1 | Batch 90/188 | Loss: 5.1521 | Acc: 1.5%
  Epoch 1 | Batch 100/188 | Loss: 5.1131 | Acc: 1.5%
  Epoch 1 | Batch 110/188 | Loss: 5.1048 | Acc: 1.5%
  Epoch 1 | Batch 120/188 | Loss: 5.0918 | Acc: 1.4%
  Epoch 1 | Batch 130/188 | Loss: 5.0872 | Acc: 1.3%
  Epoch 1 | Batch 140/188 | Loss: 5.0696 | Acc: 1.4%
  Epoch 1 | Batch 150/188 | Loss: 5.0690 | Acc: 1.4%
  Epoch 1 | Batch 160/188 | Loss: 5.0613 | Acc: 1.4%
  Epoch 1 | Batc

## Cell 10: Best Model Load karo

Training ke baad best model load karte hain inference ke liye.
Best model woh hai jisne lowest loss achieve kia.

In [10]:
# Best model load karo
model.load_state_dict(torch.load(CFG.OUTPUT_DIR / 'best_model.pth', map_location=CFG.DEVICE))
model.eval()  # Evaluation mode (Dropout off, BatchNorm fixed)

print("✅ Best model load ho gaya!")
print("   Model evaluation mode mein hai")

✅ Best model load ho gaya!
   Model evaluation mode mein hai


## Cell 11: Test Soundscapes pe Inference

**Yahan asli kaam hota hai:**
- Har test soundscape file = 1 minute ki audio
- Har 1 minute ko 12 x 5-second windows mein todte hain
- Har window ka spectrogram model se predict karte hain
- Result: probabilities for 234 species for each 5-sec window

**row_id format:**
`BC2026_Test_0001_S05_20250227_010002_20`
= File name + `_` + end time in seconds (20 = 0:15-0:20 window)

In [11]:
def predict_soundscape(filepath, model, species_cols, device=CFG.DEVICE):
    """
    Ek soundscape file (1 min) ke liye predictions banao.
    
    Returns:
        list of dicts: har 5-sec window ke liye {row_id, species: prob, ...}
    """
    results = []
    filename_stem = Path(filepath).stem  # Extension hatao
    
    try:
        # Poori 1-minute audio load karo
        audio, _ = librosa.load(filepath, sr=CFG.SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return results
    
    # 1 minute = 60 sec / 5 sec = 12 windows
    total_duration = 60
    window_size    = CFG.DURATION  # 5 seconds
    
    for start_sec in range(0, total_duration, window_size):
        end_sec = start_sec + window_size
        
        # Row ID format: filename_stem + end_time
        # e.g., BC2026_Test_0001_S05_20250227_010002_20 (end=20sec means 15-20sec window)
        row_id = f"{filename_stem}_{end_sec}"
        
        # Is window ki audio nikalo
        start_sample = start_sec * CFG.SAMPLE_RATE
        end_sample   = end_sec   * CFG.SAMPLE_RATE
        window_audio = audio[start_sample:end_sample]
        
        # Agar yeh window choti hai to pad karo
        if len(window_audio) < CFG.N_SAMPLES:
            pad = CFG.N_SAMPLES - len(window_audio)
            window_audio = np.pad(window_audio, (0, pad), mode='constant')
        else:
            window_audio = window_audio[:CFG.N_SAMPLES]
        
        # Spectrogram banao
        spec = audio_to_melspectrogram(window_audio)
        spec_tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).to(device)
        # unsqueeze(0) = batch dimension add karo: (3,H,W) → (1,3,H,W)
        
        # Model se predict karo
        with torch.no_grad():   # Gradient calculate mat karo (faster)
            logits = model(spec_tensor)               # Raw scores: (1, 234)
            probs  = torch.sigmoid(logits).cpu().numpy()[0]  # 0-1 probabilities
        
        # Result dict banao
        row = {'row_id': row_id}
        for i, col in enumerate(species_cols):
            row[col] = float(probs[i])
        
        results.append(row)
    
    return results


# Saari test files pe prediction karo
test_files = sorted(CFG.TEST_DIR.glob('*.ogg'))
print(f"📁 Test soundscape files: {len(test_files)}")

if len(test_files) == 0:
    print("⚠️  Test files nahin mili — sirf submit karne pe available hongi")
    print("   Submission file sample_submission se banate hain (placeholder)")
    all_predictions = []
else:
    all_predictions = []
    for i, test_file in enumerate(test_files):
        preds = predict_soundscape(test_file, model, SPECIES_COLS)
        all_predictions.extend(preds)
        
        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/{len(test_files)} files...")
    
    print(f"\n✅ Inference complete!")
    print(f"   Total predictions: {len(all_predictions)} rows")

📁 Test soundscape files: 0
⚠️  Test files nahin mili — sirf submit karne pe available hongi
   Submission file sample_submission se banate hain (placeholder)


## Cell 12: Submission File Banana

Ab predictions ko submission.csv mein save karte hain.

**Format:**
- Pehla column: `row_id`
- Baaki 234 columns: har species ki probability (0 to 1)
- File naam: `submission.csv` (yahi naam required hai)

Agar test files nahin mili (local testing mein hota hai), to sample_submission se format copy karte hain.

In [12]:
if len(all_predictions) > 0:
    # Predictions se DataFrame banao
    submission = pd.DataFrame(all_predictions)
    
    # Ensure sare columns sahi order mein hain
    expected_cols = ['row_id'] + SPECIES_COLS
    submission = submission.reindex(columns=expected_cols, fill_value=0.0)

else:
    # Test files nahin mili to sample_submission use karo
    # Kaggle pe actual submit karne pe test files automatically aa jayengi
    print("⚠️  Test files nahin mili — sample submission use kar rahe hain")
    submission = sample_sub.copy()
    
    # Sare species columns ko 0.5 karo (placeholder)
    for col in SPECIES_COLS:
        submission[col] = 0.5


# CSV save karo
save_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(save_path, index=False)

print(f"✅ Submission file saved!")
print(f"   Path: {save_path}")
print(f"   Shape: {submission.shape}")
print(f"   Rows: {len(submission):,}")
print(f"   Columns: {len(submission.columns)} (1 row_id + {len(SPECIES_COLS)} species)")
print(f"\nPehli 3 rows:")
submission.head(3)

⚠️  Test files nahin mili — sample submission use kar rahe hain
✅ Submission file saved!
   Path: /kaggle/working/submission.csv
   Shape: (3, 235)
   Rows: 3
   Columns: 235 (1 row_id + 234 species)

Pehli 3 rows:


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,BC2026_Test_0001_S05_20250227_010002_10,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,BC2026_Test_0001_S05_20250227_010002_15,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


## Cell 13: Final Validation

Submit karne se pehle check karo ke submission file bilkul sahi format mein hai.

**Checks:**
1. File exist karti hai
2. 'row_id' column hai
3. Sari 234 species columns hain
4. Koi missing values nahin
5. Probabilities 0 aur 1 ke darmiyan hain

In [13]:
# Final validation checks
print("🔍 Final Validation:")
print("-" * 40)

sub = pd.read_csv(CFG.OUTPUT_DIR / 'submission.csv')

# Check 1: File readable hai
print(f"✅ File readable | Shape: {sub.shape}")

# Check 2: row_id column hai
assert 'row_id' in sub.columns, "❌ row_id column missing!"
print(f"✅ row_id column present")

# Check 3: Sari species hain
missing_species = set(SPECIES_COLS) - set(sub.columns)
extra_species   = set(sub.columns) - set(SPECIES_COLS) - {'row_id'}
assert len(missing_species) == 0, f"❌ Missing species: {missing_species}"
print(f"✅ Sari {len(SPECIES_COLS)} species columns present")

# Check 4: No NaN values
nan_count = sub.isnull().sum().sum()
assert nan_count == 0, f"❌ {nan_count} NaN values hain!"
print(f"✅ No missing values")

# Check 5: Probabilities range
species_vals = sub[SPECIES_COLS].values
min_val = species_vals.min()
max_val = species_vals.max()
print(f"✅ Probability range: [{min_val:.4f}, {max_val:.4f}]")

# Summary
print("\n" + "=" * 40)
print("🎉 Submission file bilkul sahi hai!")
print(f"   File: /kaggle/working/submission.csv")
print(f"   Rows: {len(sub):,}")
print(f"   Species: {len(SPECIES_COLS)}")
print("=" * 40)
print("\n➡️  Ab Kaggle pe 'Submit' button dabao!")

🔍 Final Validation:
----------------------------------------
✅ File readable | Shape: (3, 235)
✅ row_id column present
✅ Sari 234 species columns present
✅ No missing values
✅ Probability range: [0.5000, 0.5000]

🎉 Submission file bilkul sahi hai!
   File: /kaggle/working/submission.csv
   Rows: 3
   Species: 234

➡️  Ab Kaggle pe 'Submit' button dabao!
